<h1>MSPR - BigData<h1\>



In [ ]:
from google.colab import drive

file_path = '/content/drive/MyDrive/MSPR2 - Big Data/data'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r "/content/drive/MyDrive/MSPR2 - Big Data/data" .

In [ ]:
!pip install mysql-connector-python

Prédire le résultat du bord politique (2002,2007,2012)


In [ ]:
import pandas as pd
import mysql.connector
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from datetime import datetime, timedelta
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


user = 'superadmin'
password = 'superadmin'
host = 'madeshu.fr'
database = 'MSPR2ETL_All'


conn = mysql.connector.connect(host=host, user=user, password=password, database=database)

query = """SELECT
  taux_criminalite,
  bord_nom,
  taux_chomage,
  total_resultat,
  annee
FROM (
  SELECT
    fact_criminalite.taux_criminalite,
    candidat.bord_nom,
    fact_chomage.taux_chomage,
    SUM(resultats.nombre_vote) AS total_resultat,
    annee.annee,
    ROW_NUMBER() OVER (PARTITION BY annee.annee ORDER BY SUM(resultats.nombre_vote) DESC) AS rank_num
  FROM
    fact_criminalite
    JOIN dim_annee AS annee ON fact_criminalite.annee_id = annee.annee_id
    JOIN fact_chomage ON annee.annee_id = fact_chomage.annee_id
    JOIN fact_election AS elec ON annee.annee = YEAR(elec.election_date)
    JOIN fact_resultat_candidat AS resultats ON elec.election_id = resultats.election_id
    JOIN dim_candidat AS candidat ON resultats.candidat_id = candidat.candidat_id
    WHERE
    annee.annee IN (1988,1995,2002,2007)
  GROUP BY
    candidat.bord_nom,
    fact_criminalite.taux_criminalite,
    fact_chomage.taux_chomage,
    annee.annee
) AS ranked_data
WHERE
  rank_num = 1
ORDER BY
  annee ASC"""

df = pd.read_sql(query, con=conn)
print(df)


   taux_criminalite bord_nom  taux_chomage  total_resultat  annee
0         55.974807   Gauche         8.775      10094413.0   1988
1         63.465959   Gauche        10.025       6914954.0   1995
2         68.925526    Droit         7.900       8238153.0   2002
3         58.083650    Droit         8.000      11448663.0   2007


<ipython-input-180-ef3f03a0bd50>:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn)


In [ ]:
X = df[['taux_chomage', 'taux_criminalite']]
y_encoded = pd.get_dummies(df['bord_nom'])
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)
print("Linear Regression - Mean Squared Error (MSE):", mse_linear)
print("Linear Regression - R-squared (R²):", r2_linear)


rf_model = RandomForestRegressor()
rf_model.fit(X_train, y_train)


y_pred_rf = rf_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print("Random Forest - Mean Squared Error (MSE):", mse_rf)
print("Random Forest - R-squared (R²):", r2_rf)

winning_party_rf = y_encoded.columns[y_pred_rf.argmax()]

print("Parti Gagnant :", winning_party_rf)
conn.close()


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:918: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Linear Regression - Mean Squared Error (MSE): 3.0480810750930076
Linear Regression - R-squared (R²): nan
Random Forest - Mean Squared Error (MSE): 0.4096
Random Forest - R-squared (R²): nan
Parti Gagnant : Droit


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:918: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Prédire le résultat le **taux de creminalité** sur toutes les années


> Bloc en retrait


In [ ]:
import pandas as pd
import mysql.connector
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import numpy as np
from pydataset import data
from sklearn import datasets

import statsmodels.formula.api as smf
import statsmodels.api as sm

user = 'superadmin'
password = 'superadmin'
host = 'madeshu.fr'
database = 'MSPR2ETL_All'

conn = mysql.connector.connect(host=host, user=user, password=password, database=database)

query = """
   SELECT SUM(fact_resultat_candidat.nombre_vote) as 'Total de vote',dim_candidat.bord_nom,fact_actifs.nombre_actifs,fact_chomage.taux_chomage, fact_criminalite.taux_criminalite, fact_pauvrete.taux_seuil50,fact_retraites.nb_retraites FROM fact_actifs,fact_chomage,fact_criminalite,fact_pauvrete,fact_retraites,dim_candidat,fact_resultat_candidat,fact_election,dim_annee WHERE fact_actifs.annee_id = fact_chomage.annee_id and fact_chomage.annee_id = fact_criminalite.annee_id and fact_criminalite.annee_id = fact_criminalite.annee_id and fact_criminalite.annee_id = fact_retraites.annee_id and dim_annee.annee_id = fact_retraites.annee_id and fact_retraites.annee_id = fact_pauvrete.annee_id  and fact_election.election_id = fact_resultat_candidat.election_id and fact_resultat_candidat.candidat_id = dim_candidat.candidat_id GROUP BY dim_candidat.bord_nom ORDER BY fact_resultat_candidat.nombre_vote DESC;
"""

df = pd.read_sql(query, con=conn)

conn.close()

df_prediction= pd.read_sql(query, con=conn)

conn.close()

X = df[['nombre_vote', 'nombre_actifs', 'taux_chomage', 'taux_criminalite', 'taux_seuil50', 'nb_retraites']]
y = df['bord_nom']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

regression_model = LinearRegression()
regression_model.fit(X_train, y_train)

y_pred = regression_model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Linear Regression - Mean Squared Error (MSE):", mse)
print("Linear Regression - R-squared (R²):", r2)
print("Prédictions pour les élections de 2017:")
print(df_prediction)



ModuleNotFoundError: ignored